# Native Ray head-VM-loss pilot

Run this notebook yourself with the **ray-dev (Python 3.10)** kernel. It has not been executed by the assistant. Existing notebooks are unchanged.

This tests actual head-VM stop with inaccessible head storage, surviving Redis/GCS proxy/NFS on w4, and a replacement head on w3. It compares Fixed-R OFF/ON with the same infrastructure, plus a plain healthy baseline. Failed observations are retained. This is a gated normal-task recovery diagnostic, not a demonstrated application speedup.

Read [the experiment guide](head_vm_loss/NATIVE_PILOT.md) for dependency placement, failure scope and metric limitations. **Run cells individually**, not Run All: preparation changes the cluster, and the run cell stops the head VM. Do not run other cluster experiments concurrently.


## 1. Editable paths

Use your existing five-VM state JSON. No VMs or networks are created. Relative state-file paths are resolved against the checkout. Keep the same suite name when resuming; use a new name for a new replicate.


In [ ]:
from pathlib import Path
import json
import signal
import subprocess
import sys

REPO = Path("/home/tejas/Downloads/ray")
STATE_FILE = Path("/absolute/path/to/your/rayexp-...-state.json")  # EDIT
SUITE = "native-head-v1"

REPO = REPO.expanduser().resolve()
STATE_FILE = STATE_FILE.expanduser()
if not STATE_FILE.is_absolute():
    STATE_FILE = REPO / STATE_FILE
STATE_FILE = STATE_FILE.resolve()
SCRIPT = REPO / "gossip_benchmarks/head_vm_loss/native_pilot.py"
assert SCRIPT.is_file(), f"Pilot script missing: {SCRIPT}"
assert STATE_FILE.is_file(), f"Set STATE_FILE to your existing state JSON: {STATE_FILE}"
state = json.loads(STATE_FILE.read_text())
RESULT_DIR = REPO / "ray_gcp_results/head-vm-loss" / state["run_id"] / SUITE
REPORT = RESULT_DIR / "native-report.json"
print("Kernel:", sys.executable)
print("Project:", state["project"], "Zone:", state["zone"])
print("VMs:", state["names"])
print("Report:", REPORT)


## 2. Visible command helper

This launches the plain Python harness in a fresh process, streams output into the notebook, and avoids stale module imports from other notebooks. It does not run anything until called below. Ctrl+C requests graceful supervisor cleanup; inspect the report and lock if cleanup fails.


In [ ]:
def pilot(action):
    command = [sys.executable, "-u", str(SCRIPT), action,
               "--state", str(STATE_FILE), "--suite", SUITE]
    print("Running:", command)
    process = subprocess.Popen(command, cwd=REPO, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
        status = process.wait()
    except KeyboardInterrupt:
        process.send_signal(signal.SIGINT)
        print("Interrupt requested; waiting up to 100 seconds for cleanup.", flush=True)
        try:
            process.wait(timeout=100)
        except subprocess.TimeoutExpired:
            print("Supervisor has not exited. Inspect it and the cluster lock before resuming.")
        raise
    finally:
        if process.poll() is not None:
            process.stdout.close()
    if status:
        raise RuntimeError(f"Pilot exited {status}. Read the output and saved report before proceeding.")


## 3. Inventory — read only

Inspect current instances before preparation. A saved state file does not mean the VMs are still running. Preparation requires all five recorded VMs RUNNING and the image identity expected by the existing harness.


In [ ]:
pilot("inventory")


## 4. Prepare — once for the initial setup

Moves experiment NFS off the head, installs Redis/socat, checks image identities, and uploads visible helper files. Preparation has its own 28-minute budget. **Skip this cell on an ordinary resume**; the final fault leaves the original head stopped. Do not silently restore or remove locks from a failed run.


In [ ]:
pilot("prepare")


## 5. Run or resume

Five observations: plain healthy, OFF healthy, ON healthy, OFF head loss, ON head loss. Each invocation allows 28 minutes of work plus up to 90 seconds for cleanup. The whole matrix may require more than one invocation. Rerun only this cell to continue remaining observations. Recorded failures/timeouts are not retried or overwritten.


In [ ]:
pilot("run")


## 6. Inspect saved progress — local files only

You can run this after a failed or interrupted command. A pre-injection error is not evidence of failed recovery. A completed matrix is not necessarily a successful recovery result.


In [ ]:
if not REPORT.exists():
    print("No report yet:", REPORT)
else:
    report = json.loads(REPORT.read_text())
    rows = report.get("samples", [])
    print(f"Recorded observations: {len(rows)}/5")
    for row in rows:
        result = row.get("result", {})
        print(row["label"], row["status"],
              "correct=", result.get("correct", False),
              "head confirmed off=", row.get("original_head_remained_off", False),
              "elapsed_s=", result.get("elapsed_s"))
        if row.get("error") or result.get("error"):
            print("  Error:", row.get("error") or result.get("error"))
    print("Pause reason:", report.get("paused_reason"))
    print("Cleanup errors:", report.get("cleanup_errors", []))


## 7. Collect remote evidence

Downloads the NFS evidence from surviving w4 without restarting the old head or rerunning observations. Keep this alongside the local native-report.json. Collection has separate transfer time and is not part of workload timing.


In [ ]:
pilot("download")


## 8. Plot and summarize — independent, local only

This cell can be run by itself in a fresh kernel: edit REPORT_PATH below. It makes no cloud calls and imports no project modules. Only correct completed jobs get timing points; failures remain labeled. No bars or bottom annotations. Healthy overhead excludes supervisor gate wait; single-replicate timing changes are not statistically established gains.


In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt

REPORT_PATH = Path("/home/tejas/Downloads/ray/ray_gcp_results/head-vm-loss/RUN_ID/native-head-v1/native-report.json")  # EDIT for standalone use
if "REPORT" in globals():
    REPORT_PATH = REPORT  # Uses section 1 when it was run.
report = json.loads(REPORT_PATH.read_text())
rows = report.get("samples", [])
by_label = {row["label"]: row for row in rows}

for baseline, candidate, label in [
    ("plain-healthy", "off-healthy", "Redis/proxy runtime overhead"),
    ("off-healthy", "on-healthy", "Native Fixed-R overhead"),
    ("plain-healthy", "on-healthy", "Combined enablement overhead"),
]:
    a = by_label.get(baseline, {}).get("result", {})
    b = by_label.get(candidate, {}).get("result", {})
    av = a.get("elapsed_excluding_supervisor_gate_s")
    bv = b.get("elapsed_excluding_supervisor_gate_s")
    if a.get("correct") and b.get("correct") and av and bv is not None:
        print(f"{label}: {100 * (bv / av - 1):+.2f}% ({av:.3f}s -> {bv:.3f}s)")
    else:
        print(label + ": unavailable; needs two correct healthy results")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for i, row in enumerate(rows):
    result = row.get("result", {})
    if result.get("correct"):
        axes[0].scatter(result["elapsed_s"], i, color="tab:blue")
    else:
        axes[0].text(.02, i, row["status"], color="tab:red",
                     transform=axes[0].get_yaxis_transform(), va="center")
axes[0].set_yticks(range(len(rows)), [r["label"] for r in rows])
axes[0].invert_yaxis()
axes[0].set_title("Correct job completions")
axes[0].set_xlabel("Driver elapsed time including gate (s)")

faults = [r for r in rows if r.get("fault")]
for i, row in enumerate(faults):
    value = row.get("failure_to_resumption_upper_bound_s")
    if value is not None:
        axes[1].scatter(value, i, color="tab:orange")
    else:
        axes[1].text(.02, i, "No verified resumption", color="tab:red",
                     transform=axes[1].get_yaxis_transform(), va="center")
axes[1].set_yticks(range(len(faults)), [r["label"] for r in faults])
axes[1].invert_yaxis()
axes[1].set_title("Observed resumption upper bounds")
axes[1].set_xlabel("Time from stop request (s)")
fig.tight_layout()
output = REPORT_PATH.parent / "native-head-pilot.png"
fig.savefig(output, dpi=160)
plt.show()
print("Saved:", output)
